**Inputs and local setup**

The labelled CSVs must contain `id`, `title`, `abstract`, and `year`:

- `ukb_ground_truth_positive_labelled.csv`: positive examples (the paper used UK Biobank).
- `ukb_negative_pre2014_labelled.csv`: negative examples (the paper did not use UK Biobank).

Place them in `data/validation/` or set `UKB_VALIDATION_POSITIVE_CSV` and
`UKB_VALIDATION_NEGATIVE_CSV` to their locations. Relative paths resolve from the
repository root. Set `UKB_VALIDATION_N_POS` and `UKB_VALIDATION_N_NEG` to the
intended positive integer evaluation sample sizes; no sample sizes are assumed.
The valid pools must also leave at least three positives and two negatives for
held-out prompt examples.

Dependencies must already be installed in the notebook's Python environment:
`numpy pandas torch transformers accelerate sentencepiece tqdm sentence-transformers
scikit-learn matplotlib` (plus `bitsandbytes` for 4-bit GPU inference).
The model stage can download pretrained weights. Set `HF_TOKEN` in the environment
for gated models; the notebook never prompts for credentials or installs packages.

**Output**

Sampled evaluation data, predictions, raw model outputs, performance tables,
agreement tables and heatmaps, and rankings are saved to `output/validation/`.
Override this location with `UKB_VALIDATION_OUTPUT_DIR`.


In [ ]:
# Local paths and explicit evaluation sample sizes; checked before model imports.
import os
import sys
from pathlib import Path

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src" / "utils").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))
from utils import shared_paths as P
P.bootstrap()


def configured_path(env_name, default):
    value = os.environ.get(env_name, "").strip()
    path = Path(value).expanduser() if value else default
    return path if path.is_absolute() else P.ROOT / path


TP_PATH = configured_path(
    "UKB_VALIDATION_POSITIVE_CSV",
    P.DATA / "validation" / "ukb_ground_truth_positive_labelled.csv",
)
TN_PATH = configured_path(
    "UKB_VALIDATION_NEGATIVE_CSV",
    P.DATA / "validation" / "ukb_negative_pre2014_labelled.csv",
)
OUT_DIR = configured_path("UKB_VALIDATION_OUTPUT_DIR", P.OUTPUT / "validation")

missing = [
    name for name, path in [
        ("UKB_VALIDATION_POSITIVE_CSV", TP_PATH),
        ("UKB_VALIDATION_NEGATIVE_CSV", TN_PATH),
    ]
    if not path.is_file()
]
if missing:
    raise FileNotFoundError("Missing labelled CSVs: set " + ", ".join(missing) + ".")


def positive_sample_size(env_name):
    value = os.environ.get(env_name, "").strip()
    try:
        result = int(value)
    except ValueError:
        raise ValueError(f"Set {env_name} to the intended positive integer sample size.") from None
    if result <= 0:
        raise ValueError(f"{env_name} must be a positive integer; got {value!r}.")
    return result


N_POS = positive_sample_size("UKB_VALIDATION_N_POS")
N_NEG = positive_sample_size("UKB_VALIDATION_N_NEG")
SEED = 42
USE_4BIT = True

OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import re, json, time, gc, ast, random, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from IPython.display import display
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

# Hugging Face loaders below read HF_TOKEN directly; no interactive login required.
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:


# ============================================================
# Data loading
# ============================================================

def extract_text(x):
    if pd.isna(x):
        return ""
    if not isinstance(x, str):
        return str(x).strip()
    s = x.strip()
    if s.startswith("{") and s.endswith("}"):
        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, dict):
                if "preferred" in obj and obj["preferred"] is not None:
                    return str(obj["preferred"]).strip()
                for v in obj.values():
                    if v is not None and str(v).strip():
                        return str(v).strip()
        except Exception:
            pass
    return s

def load_labelled_csv(path, expected_label):
    df = pd.read_csv(path)

    for c in ["id", "title", "abstract", "year"]:
        if c not in df.columns:
            raise ValueError(f"Missing column {c} in {path}. Columns: {df.columns.tolist()}")

    out = df[["id", "title", "abstract", "year"]].copy()
    out["id"] = out["id"].astype(str).str.strip()
    out["title"] = out["title"].apply(extract_text)
    out["abstract"] = out["abstract"].apply(extract_text)
    out["year"] = pd.to_numeric(out["year"], errors="coerce")
    out["label"] = int(expected_label)

    out = out[out["id"].ne("")]
    out = out[out["abstract"].fillna("").astype(str).str.strip().ne("")]
    out = out.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)
    return out

tp_pool = load_labelled_csv(TP_PATH, 1)
tn_pool = load_labelled_csv(TN_PATH, 0)

print("TP pool:", len(tp_pool), "unique ids:", tp_pool["id"].nunique())
print("TN pool:", len(tn_pool), "unique ids:", tn_pool["id"].nunique())

if len(tp_pool) < N_POS + 3:
    raise ValueError(
        f"Positive pool has {len(tp_pool)} valid papers; UKB_VALIDATION_N_POS={N_POS} "
        "requires three additional papers for held-out prompt examples."
    )
if len(tn_pool) < N_NEG + 2:
    raise ValueError(
        f"Negative pool has {len(tn_pool)} valid papers; UKB_VALIDATION_N_NEG={N_NEG} "
        "requires two additional papers for held-out prompt examples."
    )

tp_eval = tp_pool.sample(n=N_POS, random_state=SEED).copy()
tn_eval = tn_pool.sample(n=N_NEG, random_state=SEED).copy()

eval_df = pd.concat([tp_eval, tn_eval], ignore_index=True)
eval_df = eval_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
eval_df["True_label"] = eval_df["label"].astype(bool)

eval_path = os.path.join(OUT_DIR, f"ukb_eval_{N_POS}pos_{N_NEG}neg.csv")
eval_df.to_csv(eval_path, index=False)

print("Eval set:", eval_df.shape)
print(eval_df["label"].value_counts())
print("Saved eval set:", eval_path)
display(eval_df.head(10))

# ============================================================
# Few-shot example selection
# ============================================================

def contains_ukb_text(row):
    text = f"{row.get('title','')} {row.get('abstract','')}".lower()
    return any(x in text for x in [
        "uk biobank", "ukb", "ukbb", "united kingdom biobank", "uk biobank resource"
    ])

def contains_hard_negative_cue(row):
    text = f"{row.get('title','')} {row.get('abstract','')}".lower()
    cues = [
        "unlike uk biobank", "compared with uk biobank", "such as uk biobank",
        "including uk biobank", "biobanks such as", "ethical", "governance",
        "china kadoorie", "biobank japan", "finnish biobank", "fingen", "finngen",
        "all of us", "janus serum bank", "copenhagen hospital biobank"
    ]
    return any(c in text for c in cues)

eval_ids = set(eval_df["id"].astype(str))
tp_demo_pool = tp_pool[~tp_pool["id"].astype(str).isin(eval_ids)].copy()
tn_demo_pool = tn_pool[~tn_pool["id"].astype(str).isin(eval_ids)].copy()

# Prefer one positive without explicit UKB text if available, because this is the hard recall case.
tp_no_ukb = tp_demo_pool[~tp_demo_pool.apply(contains_ukb_text, axis=1)]
tp_with_ukb = tp_demo_pool[tp_demo_pool.apply(contains_ukb_text, axis=1)]
tn_hard = tn_demo_pool[tn_demo_pool.apply(contains_hard_negative_cue, axis=1)]

one_pos = (tp_no_ukb if len(tp_no_ukb) > 0 else tp_demo_pool).sample(n=1, random_state=SEED + 1)
one_neg = (tn_hard if len(tn_hard) > 0 else tn_demo_pool).sample(n=1, random_state=SEED + 2)

# Five-shot: 3 positives + 2 negatives.
tp_demo_1 = (tp_no_ukb if len(tp_no_ukb) >= 1 else tp_demo_pool).sample(n=1, random_state=SEED + 3)
tp_demo_2 = (tp_with_ukb if len(tp_with_ukb) >= 2 else tp_demo_pool).sample(n=2, random_state=SEED + 4)
tn_demo_1 = (tn_hard if len(tn_hard) >= 1 else tn_demo_pool).sample(n=1, random_state=SEED + 5)
tn_demo_2 = tn_demo_pool.sample(n=1, random_state=SEED + 6)

fewshot_examples = []
for _, r in pd.concat([tp_demo_1, tp_demo_2]).iterrows():
    fewshot_examples.append({"title": r["title"], "abstract": r["abstract"], "label": True})
for _, r in pd.concat([tn_demo_1, tn_demo_2]).iterrows():
    fewshot_examples.append({"title": r["title"], "abstract": r["abstract"], "label": False})

random.Random(SEED).shuffle(fewshot_examples)

print("\nOne-shot positive example:")
display(one_pos[["id", "title", "abstract", "label"]].head(1))

print("\nOne-shot negative example:")
display(one_neg[["id", "title", "abstract", "label"]].head(1))

print("\nFew-shot examples:")
for ex in fewshot_examples:
    print(ex["label"], ex["title"][:120])

# ============================================================
# Prompts
# ============================================================

def truncate_text(s, max_chars=3500):
    s = str(s or "")
    return s[:max_chars]

def json_instruction():
    return """Return strict JSON only:
{"implies_UKB_use":true|false}
Do not include explanation, markdown, or any extra keys."""

def make_prompt_v1_conservative(title, abstract):
    return f"""You will be given a scientific paper title and abstract.

Task: decide whether the paper used UK Biobank data or resources.

Definition:
- true: the study used UK Biobank data, participants, samples, imaging, genetics, linked health records, or another UK Biobank resource.
- false: the paper only mentions UK Biobank, discusses biobanks generally, compares with UK Biobank, or uses other biobanks but not UK Biobank.

Be conservative. If uncertain, return false.

{json_instruction()}

title: \"\"\"{truncate_text(title, 700)}\"\"\"

abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"

JSON:
"""

def make_prompt_v2_balanced(title, abstract):
    return f"""You will be given a scientific paper title and abstract.

Task: decide whether the paper likely used UK Biobank data or resources.

Important:
- Some true UK Biobank-use papers do not mention "UK Biobank" in the abstract.
- The paper may still use UK Biobank if the abstract describes a UK population-scale cohort, genetic/imaging/health-record analysis, or data-resource use that is consistent with UK Biobank.
- Do not require explicit words "UK Biobank" if the evidence strongly suggests use.

Return true when the title/abstract provides reasonable evidence that the paper analysed UK Biobank participants, data, samples, imaging, genetics, linked records, or a UK Biobank-derived cohort.

Return false when the paper is only about generic biobanking, ethics/governance, reviews, comparisons, or another named biobank.

{json_instruction()}

title: \"\"\"{truncate_text(title, 700)}\"\"\"

abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"

JSON:
"""

def make_prompt_v3_evidence_cues(title, abstract):
    return f"""Classify whether this paper uses UK Biobank.

Use the following cues.

Positive evidence can include:
- explicit UK Biobank / UKB / UKBB mention;
- analysis of a very large UK cohort with genetic, imaging, health-record, lifestyle, biomarker, or hospital-linked data;
- phrases such as participants, cohort, baseline assessment, imaging assessment, genotyping, exome sequencing, linked health records, Hospital Episode Statistics, or Townsend deprivation index in a UK population context;
- a study design that clearly analyses participant-level data rather than merely discussing biobanks.

Negative evidence can include:
- generic discussion of biobanks;
- ethics, governance, consent, infrastructure, sample storage, or review articles;
- use of another biobank only;
- mentions like "such as UK Biobank", "unlike UK Biobank", or comparison with UK Biobank.

Prefer true if the abstract strongly looks like an original analysis using UK Biobank-style data, even if UK Biobank is not named in the abstract.
Prefer false if the abstract is generic or only about other biobanks.

{json_instruction()}

title: \"\"\"{truncate_text(title, 700)}\"\"\"

abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"

JSON:
"""

def make_prompt_v4_context_no_shot(title, abstract):
    return f"""You will classify a paper using only its title and abstract.

Context:
All papers in this evaluation were retrieved because their full text matched at least one UK Biobank-related query. Therefore, UK Biobank may be mentioned only outside the abstract.

Question:
Based on the title and abstract, is it likely that the paper used UK Biobank data/resources in its own analysis?

Label true if likely UK Biobank use.
Label false if the paper likely only mentions UK Biobank, discusses biobanks generally, or uses other non-UKB resources.

Do not be overly strict: if the paper is an original epidemiological, genetic, imaging, biomarker, or clinical-risk study and the abstract strongly suggests use of a UK population-scale linked cohort, return true.

{json_instruction()}

title: \"\"\"{truncate_text(title, 700)}\"\"\"

abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"

JSON:
"""

def make_prompt_v5_one_shot(title, abstract):
    pos = one_pos.iloc[0]
    neg = one_neg.iloc[0]

    return f"""You will be given a scientific paper title and abstract.

Task:
Decide whether the paper likely used UK Biobank data/resources in its own analysis.

Context:
All papers in this evaluation were retrieved because their full text matched a UK Biobank-related search. Some true positives may not mention UK Biobank in the abstract.

{json_instruction()}

Example positive:
title: \"\"\"{truncate_text(pos["title"], 500)}\"\"\"
abstract: \"\"\"{truncate_text(pos["abstract"], 1800)}\"\"\"
answer: {{"implies_UKB_use":true}}

Example negative:
title: \"\"\"{truncate_text(neg["title"], 500)}\"\"\"
abstract: \"\"\"{truncate_text(neg["abstract"], 1800)}\"\"\"
answer: {{"implies_UKB_use":false}}

Now classify this paper.

title: \"\"\"{truncate_text(title, 700)}\"\"\"

abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"

JSON:
"""

def make_prompt_v6_five_shot(title, abstract):
    examples_txt = []
    for i, ex in enumerate(fewshot_examples, start=1):
        examples_txt.append(
            f"""Example {i}
title: \"\"\"{truncate_text(ex["title"], 450)}\"\"\"
abstract: \"\"\"{truncate_text(ex["abstract"], 1300)}\"\"\"
answer: {{"implies_UKB_use":{str(ex["label"]).lower()}}}
"""
        )

    examples_block = "\n".join(examples_txt)

    return f"""You will be given a scientific paper title and abstract.

Task:
Decide whether the paper likely used UK Biobank data/resources in its own analysis.

Context:
All papers in this evaluation were retrieved because their full text matched a UK Biobank-related search.
Some true UK Biobank-use papers may not mention UK Biobank in the abstract.
Some false positives mention biobanks, UK cohorts, or other biobanks but do not use UK Biobank.

Return true when the paper likely analysed UK Biobank participants, data, samples, imaging, genetics, linked health records, or other UK Biobank resources.
Return false for generic biobank discussion, ethics/governance, reviews, comparisons, or other-biobank-only papers.

{json_instruction()}

Few-shot examples:
{examples_block}

Now classify this paper.

title: \"\"\"{truncate_text(title, 700)}\"\"\"

abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"

JSON:
"""

PROMPT_BUILDERS = {
    "p1_conservative": make_prompt_v1_conservative,
    "p2_balanced": make_prompt_v2_balanced,
    "p3_evidence_cues": make_prompt_v3_evidence_cues,
    "p4_context_no_shot": make_prompt_v4_context_no_shot,
    "p5_real_one_shot": make_prompt_v5_one_shot,
    "p6_real_five_shot": make_prompt_v6_five_shot,
}

# ============================================================
# Parsing and metrics
# ============================================================

def extract_first_json_obj(text):
    raw = (text or "").strip()
    raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw, flags=re.I | re.S).strip()
    try:
        return json.loads(raw)
    except Exception:
        pass

    m = re.search(r"\{.*?\}", raw, flags=re.S)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    return None

def parse_llm_result(obj):
    if not isinstance(obj, dict):
        return None

    val = obj.get("implies_UKB_use", None)

    if isinstance(val, bool):
        return val

    if isinstance(val, str):
        v = val.strip().lower()
        if v in {"true", "yes", "1"}:
            return True
        if v in {"false", "no", "0"}:
            return False

    return None

def build_model_input(tokenizer, user_text):
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        msgs = [{"role": "user", "content": user_text}]
        try:
            return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        except Exception:
            return user_text
    return user_text

def metric_dict(y_true, y_pred):
    y_true = np.asarray(y_true).astype(bool)
    y_pred = np.asarray(y_pred).astype(bool)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[False, True]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
    }

def pairwise_agreement_matrix(pred_df, model_cols):
    agree = pd.DataFrame(index=model_cols, columns=model_cols, dtype=float)
    nmat = pd.DataFrame(index=model_cols, columns=model_cols, dtype=int)

    for a in model_cols:
        for b in model_cols:
            m = pred_df[[a, b]].dropna()
            n = len(m)
            nmat.loc[a, b] = n
            agree.loc[a, b] = (m[a].astype(bool).values == m[b].astype(bool).values).mean() if n else np.nan

    return agree, nmat

def save_agreement_heatmap(agreement, prompt_name, out_dir):
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(agreement.values.astype(float), vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(agreement.columns)))
    ax.set_yticks(range(len(agreement.index)))
    ax.set_xticklabels(agreement.columns, rotation=45, ha="right")
    ax.set_yticklabels(agreement.index)
    ax.set_title(f"Pairwise agreement: {prompt_name}")

    for i in range(len(agreement.index)):
        for j in range(len(agreement.columns)):
            val = agreement.iloc[i, j]
            if pd.notna(val):
                ax.text(j, i, f"{val*100:.1f}", ha="center", va="center", fontsize=8)

    fig.colorbar(im, ax=ax)
    plt.tight_layout()

    path = os.path.join(out_dir, f"pairwise_agreement_heatmap_{prompt_name}.png")
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved heatmap:", path)
    return path

# ============================================================
# LLM loading and prediction
# ============================================================

def load_llm(model_id, tag):
    token = os.getenv("HF_TOKEN", None)

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        use_fast=True,
        token=token,
        trust_remote_code=True,
    )

    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    quant_config = None
    dtype = torch.float16

    if USE_4BIT and torch.cuda.is_available():
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )

    kwargs = dict(
        device_map="auto",
        torch_dtype=dtype,
        token=token,
        trust_remote_code=True,
    )

    if quant_config is not None:
        kwargs["quantization_config"] = quant_config

    # Phi-specific fix: use Phi-3.5 and eager attention.
    # This avoids the common Phi-3 rope/config/flash-attention loading issue.
    if "phi" in tag.lower():
        kwargs["attn_implementation"] = "eager"

    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs).eval()

    return tokenizer, model

def run_llm_for_prompt(model_id, tag, prompt_name, prompt_fn, df, batch_size=8, max_input_tokens=4096, max_new_tokens=64):
    print("\n" + "=" * 100)
    print(f"LLM: {tag} | Prompt: {prompt_name}")
    print(f"Model: {model_id}")
    print("=" * 100)

    tokenizer, model = load_llm(model_id, tag)

    n = len(df)
    preds = np.full(n, np.nan, dtype=object)
    raw_outputs = [""] * n
    parse_ok = np.zeros(n, dtype=bool)

    t0 = time.time()

    for s in tqdm(range(0, n, batch_size), desc=f"{tag}-{prompt_name}"):
        e = min(s + batch_size, n)
        batch = df.iloc[s:e]

        prompts = [
            build_model_input(tokenizer, prompt_fn(r["title"], r["abstract"]))
            for _, r in batch.iterrows()
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input_tokens,
        ).to(model.device)

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[1]
        gen_only = output_ids[:, prompt_len:]
        texts = tokenizer.batch_decode(gen_only, skip_special_tokens=True)

        for i, txt in enumerate(texts):
            idx = s + i
            raw_outputs[idx] = txt
            obj = extract_first_json_obj(txt)
            val_out = parse_llm_result(obj)

            if val_out is not None:
                preds[idx] = bool(val_out)
                parse_ok[idx] = True

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    runtime = time.time() - t0

    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return preds, raw_outputs, {
        "model": tag,
        "prompt": prompt_name,
        "type": "LLM",
        "runtime_s": runtime,
        "items_per_sec": n / max(runtime, 1e-9),
        "parse_rate": parse_ok.mean(),
        "n_parsed": int(parse_ok.sum()),
    }

# ============================================================
# Encoder baselines
# ============================================================

def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    return (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

def best_threshold(scores, labels):
    labels = np.asarray(labels).astype(bool)
    scores = np.asarray(scores)
    thresholds = np.linspace(scores.min(), scores.max(), 200)

    best_f1 = -1
    best_t = None

    for t in thresholds:
        pred = scores >= t
        f1 = f1_score(labels, pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return float(best_t), float(best_f1)

def run_scibert_baseline(df):
    tag = "scibert_sim"
    model_id = "allenai/scibert_scivocab_uncased"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("\n" + "=" * 100)
    print("Encoder baseline:", tag)
    print("=" * 100)

    tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    model = AutoModel.from_pretrained(model_id).to(device).eval()

    query = "This scientific paper uses UK Biobank data or resources."
    q = tok([query], return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

    with torch.inference_mode():
        qout = model(**q)
        qemb = mean_pool(qout.last_hidden_state, q["attention_mask"])
        qemb = torch.nn.functional.normalize(qemb, dim=1)

    scores = np.zeros(len(df))
    texts = (df["title"].fillna("") + "\n" + df["abstract"].fillna("")).tolist()

    t0 = time.time()
    batch_size = 64

    for s in tqdm(range(0, len(texts), batch_size), desc=tag):
        e = min(s + batch_size, len(texts))
        inp = tok(texts[s:e], return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

        with torch.inference_mode():
            out = model(**inp)
            emb = mean_pool(out.last_hidden_state, inp["attention_mask"])
            emb = torch.nn.functional.normalize(emb, dim=1)
            scores[s:e] = (emb @ qemb.T).squeeze(1).detach().cpu().numpy()

    runtime = time.time() - t0

    del model, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    thr, calib_f1 = best_threshold(scores, df["True_label"].values)
    preds = scores >= thr

    meta = {
        "model": tag,
        "prompt": "encoder_similarity",
        "type": "Encoder",
        "runtime_s": runtime,
        "items_per_sec": len(df) / max(runtime, 1e-9),
        "threshold": thr,
        "calib_f1_all_eval": calib_f1,
    }

    return preds, scores, meta

def run_sbert_baseline(df):
    tag = "sbert_minilm_sim"
    model_id = "sentence-transformers/all-MiniLM-L6-v2"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("\n" + "=" * 100)
    print("Encoder baseline:", tag)
    print("=" * 100)

    model = SentenceTransformer(model_id, device=device)

    query = "This scientific paper uses UK Biobank data or resources."
    q_emb = model.encode([query], normalize_embeddings=True)

    texts = (df["title"].fillna("") + "\n" + df["abstract"].fillna("")).tolist()

    t0 = time.time()
    emb = model.encode(texts, normalize_embeddings=True, batch_size=128, show_progress_bar=True)
    runtime = time.time() - t0

    scores = emb @ q_emb[0]

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    thr, calib_f1 = best_threshold(scores, df["True_label"].values)
    preds = scores >= thr

    meta = {
        "model": tag,
        "prompt": "encoder_similarity",
        "type": "Encoder",
        "runtime_s": runtime,
        "items_per_sec": len(df) / max(runtime, 1e-9),
        "threshold": thr,
        "calib_f1_all_eval": calib_f1,
    }

    return preds, scores, meta

# ============================================================
# Models
# ============================================================

LLM_SPECS = [
    ("Qwen/Qwen2.5-7B-Instruct", "qwen2_5_7b", 8),
    ("meta-llama/Meta-Llama-3-8B-Instruct", "llama3_8b", 8),
    ("mistralai/Mistral-7B-Instruct-v0.3", "mistral_7b", 8),
    ("microsoft/Phi-3.5-mini-instruct", "phi3_5_mini", 16),
    ("HuggingFaceH4/zephyr-7b-beta", "zephyr_7b", 8),
]

if not os.getenv("HF_TOKEN", "").strip():
    LLM_SPECS = [x for x in LLM_SPECS if "llama" not in x[1]]
    print("Skipping Llama-3 because no HF token/access was detected.")

print("LLMs to run:")
for model_id, tag, bs in LLM_SPECS:
    print(tag, model_id, "batch_size=", bs)

# ============================================================
# Run prompt-by-prompt
# ============================================================

all_results = []
all_prediction_files = []

y_true = eval_df["True_label"].astype(bool).values

for prompt_name, prompt_fn in PROMPT_BUILDERS.items():
    print("\n\n" + "#" * 120)
    print(f"RUNNING PROMPT: {prompt_name}")
    print("#" * 120)

    preds_wide = eval_df[["id", "title", "abstract", "year", "True_label"]].copy()
    raw_df = eval_df[["id", "True_label"]].copy()
    prompt_results = []

    for model_id, tag, batch_size in LLM_SPECS:
        try:
            preds, raw_outputs, meta = run_llm_for_prompt(
                model_id=model_id,
                tag=tag,
                prompt_name=prompt_name,
                prompt_fn=prompt_fn,
                df=eval_df,
                batch_size=batch_size,
                max_input_tokens=4096,
                max_new_tokens=80,
            )

            preds_wide[tag] = preds
            raw_df[f"{tag}_raw"] = raw_outputs

            mask = pd.notna(preds)
            if mask.sum() > 0:
                metrics = metric_dict(y_true[mask], np.asarray(preds[mask], dtype=bool))
            else:
                metrics = {
                    "accuracy": np.nan, "precision": np.nan, "recall": np.nan, "f1": np.nan,
                    "tp": 0, "tn": 0, "fp": 0, "fn": 0,
                }

            row = {**meta, **metrics}
            prompt_results.append(row)
            all_results.append(row)

            print(f"\nResult for {tag} / {prompt_name}")
            display(pd.DataFrame([row]))

        except Exception as e:
            print(f"[ERROR] {tag} failed for {prompt_name}: {type(e).__name__}: {e}")
            row = {
                "model": tag,
                "prompt": prompt_name,
                "type": "LLM",
                "error": f"{type(e).__name__}: {e}",
            }
            prompt_results.append(row)
            all_results.append(row)

    try:
        scibert_preds, scibert_scores, meta = run_scibert_baseline(eval_df)
        preds_wide["scibert_sim"] = scibert_preds
        preds_wide["scibert_score"] = scibert_scores
        row = {**meta, **metric_dict(y_true, scibert_preds)}
        row["prompt"] = prompt_name
        prompt_results.append(row)
        all_results.append(row)
    except Exception as e:
        print("[ERROR] SciBERT failed:", e)

    try:
        sbert_preds, sbert_scores, meta = run_sbert_baseline(eval_df)
        preds_wide["sbert_minilm_sim"] = sbert_preds
        preds_wide["sbert_score"] = sbert_scores
        row = {**meta, **metric_dict(y_true, sbert_preds)}
        row["prompt"] = prompt_name
        prompt_results.append(row)
        all_results.append(row)
    except Exception as e:
        print("[ERROR] S-BERT failed:", e)

    pred_path = os.path.join(OUT_DIR, f"predictions_{prompt_name}.csv")
    raw_path = os.path.join(OUT_DIR, f"raw_outputs_{prompt_name}.csv")
    res_path = os.path.join(OUT_DIR, f"results_{prompt_name}.csv")

    preds_wide.to_csv(pred_path, index=False)
    raw_df.to_csv(raw_path, index=False)

    res_df_prompt = pd.DataFrame(prompt_results)
    res_df_prompt.to_csv(res_path, index=False)

    print("\n=== Results for prompt:", prompt_name, "===")
    display(res_df_prompt.sort_values("accuracy", ascending=False, na_position="last"))

    model_cols = [
        c for c in preds_wide.columns
        if c not in ["id", "title", "abstract", "year", "True_label", "scibert_score", "sbert_score"]
    ]

    agreement, nmat = pairwise_agreement_matrix(preds_wide, model_cols)
    agreement_pct = (agreement * 100).round(1)

    print("\n=== Pairwise agreement (%) for prompt:", prompt_name, "===")
    display(agreement_pct)

    print("\n=== Pairwise compared-row counts for prompt:", prompt_name, "===")
    display(nmat)

    agreement_path = os.path.join(OUT_DIR, f"pairwise_agreement_percent_{prompt_name}.csv")
    nmat_path = os.path.join(OUT_DIR, f"pairwise_agreement_n_{prompt_name}.csv")

    agreement_pct.to_csv(agreement_path)
    nmat.to_csv(nmat_path)

    heatmap_path = save_agreement_heatmap(agreement, prompt_name, OUT_DIR)

    print("Saved:")
    print(pred_path)
    print(raw_path)
    print(res_path)
    print(agreement_path)
    print(nmat_path)
    print(heatmap_path)

    all_prediction_files.append(pred_path)

# ============================================================
# Combined summary and final ranking
# ============================================================

all_results_df = pd.DataFrame(all_results)
all_results_path = os.path.join(OUT_DIR, "ALL_prompt_model_results_summary.csv")
all_results_df.to_csv(all_results_path, index=False)

print("\nSaved combined results:", all_results_path)

metric_cols = [
    "prompt", "model", "type",
    "accuracy", "precision", "recall", "f1",
    "tp", "tn", "fp", "fn",
    "parse_rate", "items_per_sec",
    "runtime_s", "error"
]
existing_cols = [c for c in metric_cols if c in all_results_df.columns]

ranked = (
    all_results_df[existing_cols]
    .sort_values(["accuracy", "f1", "precision", "recall"], ascending=[False, False, False, False], na_position="last")
    .reset_index(drop=True)
)

ranked_path = os.path.join(OUT_DIR, "ALL_prompt_model_results_ranked_by_accuracy.csv")
ranked.to_csv(ranked_path, index=False)

print("\n=== FINAL ORDERED RESULTS: highest accuracy to lowest ===")
display(ranked)

print("Saved ranked results:", ranked_path)

# Optional: show best LLM-only results
if "type" in ranked.columns:
    print("\n=== LLM-only ranking ===")
    display(ranked[ranked["type"].eq("LLM")].reset_index(drop=True))

In [ ]:

from pathlib import Path
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



# Use the configured output directory from the setup cell.
OUT_DIR = Path(OUT_DIR)

PROMPTS = [
    ("p1_conservative", "a.", "Conservative instructions"),
    ("p2_balanced", "b.", "Balanced instructions"),
    ("p3_evidence_cues", "c.", "Evidence cues"),
    ("p4_context_no_shot", "d.", "Context, no-shot"),
    ("p5_real_one_shot", "e.", "Real one-shot"),
    ("p6_real_five_shot", "f.", "Real five-shot"),
]

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.ravel()

shared_image = None

for ax, (prompt_name, panel_tag, panel_title) in zip(axes, PROMPTS):
    agreement_path = (
        OUT_DIR / f"pairwise_agreement_percent_{prompt_name}.csv"
    )

    if not agreement_path.exists():
        raise FileNotFoundError(
            f"Agreement table not found: {agreement_path}"
        )

    agreement = pd.read_csv(agreement_path, index_col=0)
    agreement = agreement.apply(pd.to_numeric, errors="coerce")

    # Saved tables use percentages; imshow in the original notebook uses 0-1.
    if agreement.max().max() > 1:
        agreement = agreement / 100

    shared_image = ax.imshow(
        agreement.values.astype(float),
        vmin=0,
        vmax=1,
        aspect="auto",
    )

    ax.set_xticks(range(len(agreement.columns)))
    ax.set_yticks(range(len(agreement.index)))
    ax.set_xticklabels(
        agreement.columns,
        rotation=45,
        ha="right",
        fontsize=8.5,
    )
    ax.set_yticklabels(agreement.index, fontsize=8.5)
    ax.set_title(panel_title, fontsize=12, pad=13)

    ax.text(
        -0.12,
        1.08,
        panel_tag,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        ha="left",
        va="top",
    )

    for row_index in range(len(agreement.index)):
        for column_index in range(len(agreement.columns)):
            value = agreement.iloc[row_index, column_index]

            if pd.notna(value):
                ax.text(
                    column_index,
                    row_index,
                    f"{value * 100:.1f}",
                    ha="center",
                    va="center",
                    fontsize=7.5,
                    color="black",
                )

# Use one shared colour bar for all six panels.
colourbar_axis = fig.add_axes([0.925, 0.19, 0.014, 0.67])
colourbar = fig.colorbar(shared_image, cax=colourbar_axis)
colourbar.set_label("Pairwise agreement", fontsize=10)
colourbar.set_ticks(np.linspace(0, 1, 6))
colourbar.set_ticklabels(
    [f"{value:.0%}" for value in np.linspace(0, 1, 6)]
)

caption = (
    "Fig. 1 | Pairwise agreement across the six prompt strategies and "
    "six model/baseline configurations. The three instruction-tuned "
    "deployment LLMs showed consistently high mutual agreement across "
    "prompts, whereas encoder baselines behaved differently and were "
    "less suitable for final corpus construction."
)

fig.text(
    0.5,
    0.025,
    textwrap.fill(caption, width=155),
    ha="center",
    va="bottom",
    fontsize=11,
)

fig.subplots_adjust(
    left=0.08,
    right=0.90,
    top=0.95,
    bottom=0.17,
    wspace=0.42,
    hspace=0.55,
)

png_path = OUT_DIR / "fig1_combined_pairwise_agreement_heatmaps.png"
pdf_path = OUT_DIR / "fig1_combined_pairwise_agreement_heatmaps.pdf"

fig.savefig(png_path, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")

plt.show()

print(f"Saved PNG: {png_path}")
print(f"Saved PDF: {pdf_path}")